# LeetCode 3547 — Maximum Sum of Edge Values in a Graph bằng Python

- Link: https://leetcode.com/problems/maximum-sum-of-edge-values-in-a-graph/
- Độ khó: **Hard**
- Chủ đề: **Đồ thị bậc ≤ 2 (đường hoặc chu trình)**, **tham lam sắp xếp xen kẽ**, **lập luận trao đổi** (exchange argument), công thức đóng bằng tổng lũy thừa

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxScore(self, n: int, edges: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra đồ thị **liên thông, mọi đỉnh bậc ≤ 2** chỉ có thể là **một đường đi** (`m = n − 1`) hoặc **một chu trình** (`m = n`).
2. Biến bài toán thành: xếp các số `1..n` lên một vòng (hoặc một dãy) để **tổng tích hai số kề nhau** lớn nhất.
3. Hiểu cách xếp tối ưu "xen kẽ quanh số lớn nhất": `… 5, 3, 1, 2, 4, 6 …` — số lớn đứng cạnh số lớn.
4. Chứng minh bằng **lập luận trao đổi** (đảo một đoạn: `(a−d)(c−b)`), kiểm chứng bằng vét cạn hoán vị.
5. Rút ra **công thức đóng** `O(1)` bằng tổng `Σ i(i+2)`.

## 2. Đề bài & ví dụ

Đồ thị vô hướng liên thông `n` đỉnh, mỗi đỉnh nối với **tối đa 2** đỉnh khác; `m` cạnh `edges[i] = [a, b]`. Gán cho mỗi đỉnh một giá trị **khác nhau** trong `1..n`. Giá trị một cạnh = **tích** giá trị hai đầu. Điểm = tổng giá trị mọi cạnh. Trả về điểm lớn nhất.

**Ví dụ 1:** `n = 4`, `edges = [[0,1],[1,2],[2,3]]` (đường đi) → `23` (gán theo đường: `1, 3, 4, 2`: `3 + 12 + 8 = 23`).

**Ví dụ 2:** `n = 6`, `edges = [[0,3],[4,5],[2,0],[1,3],[2,4],[1,5]]` (chu trình) → `82` (`1·2 + 2·4 + 4·6 + 6·5 + 5·3 + 3·1`).

**Ràng buộc:** `1 ≤ n ≤ 5·10^4`, `1 ≤ m ≤ n`, không cạnh lặp, liên thông, bậc ≤ 2.

In [1]:
from typing import List
examples = [
    (4, [[0, 1], [1, 2], [2, 3]], 23),
    (6, [[0, 3], [4, 5], [2, 0], [1, 3], [2, 4], [1, 5]], 82),
]
for n, e, exp in examples:
    print(n, e, '-> kỳ vọng', exp)

4 [[0, 1], [1, 2], [2, 3]] -> kỳ vọng 23
6 [[0, 3], [4, 5], [2, 0], [1, 3], [2, 4], [1, 5]] -> kỳ vọng 82


## 3. Kiến thức nền

### 3.1. Đồ thị bậc ≤ 2 liên thông

Đồ thị liên thông `n` đỉnh có ít nhất `n − 1` cạnh. Nếu mọi bậc ≤ 2 thì tổng bậc ≤ `2n` ⇒ `m ≤ n`. Vậy:

- `m = n − 1`: một **cây** với bậc ≤ 2 ⇒ **đường đi**.
- `m = n`: đúng một chu trình, và vì bậc ≤ 2 nên toàn bộ là **một chu trình**.

Do đó cấu trúc cạnh cụ thể không quan trọng — chỉ cần biết `m == n` hay không.

In [2]:
for n, e, _ in examples:
    deg = [0] * n
    for a, b in e:
        deg[a] += 1; deg[b] += 1
    print(f"n={n}, m={len(e)} -> {'chu trình' if len(e) == n else 'đường đi'}, bậc = {deg}")

n=4, m=3 -> đường đi, bậc = [1, 2, 2, 1]
n=6, m=6 -> chu trình, bậc = [2, 2, 2, 2, 2, 2]


### 3.2. Tính điểm của một cách xếp

Với cách xếp `order` (dãy giá trị dọc đường/vòng): điểm đường đi = `Σ order[i]·order[i+1]`; chu trình cộng thêm `order[-1]·order[0]`.

In [3]:
def score_of(order, cycle):
    s = sum(order[i] * order[i + 1] for i in range(len(order) - 1))
    if cycle and len(order) > 2:
        s += order[-1] * order[0]
    return s

print(score_of([1, 3, 4, 2], False), score_of([1, 2, 4, 6, 5, 3], True))

23 82


### 3.3. `collections.deque`: thêm vào hai đầu

`appendleft` / `append` đều `O(1)` — tiện để xếp số từ lớn đến nhỏ, luân phiên đặt bên trái và bên phải.

In [4]:
from collections import deque
dq = deque([6])
for k, x in enumerate([5, 4, 3, 2, 1]):
    (dq.append if k % 2 == 0 else dq.appendleft)(x)
print(list(dq))

[2, 4, 6, 5, 3, 1]


### 3.4. Công thức tổng

`Σ_{i=1}^{k} i = k(k+1)/2`, `Σ_{i=1}^{k} i² = k(k+1)(2k+1)/6`. Dùng để rút gọn `Σ i(i+2) = Σ i² + 2Σ i`.

In [5]:
k = 10
print(sum(i * (i + 2) for i in range(1, k + 1)), k * (k + 1) * (2 * k + 1) // 6 + k * (k + 1))

495 495


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Muốn tổng tích kề lớn ⇒ số **lớn** nên đứng cạnh số **lớn**. Số lớn nhất `n` có hai hàng xóm (trên vòng) ⇒ cho nó `n−1` và `n−2`. Tiếp theo `n−1` còn một chỗ trống ⇒ cho `n−3`; `n−2` còn chỗ ⇒ cho `n−4`; … Kết quả là dãy "hình núi":

`1, 3, 5, …, n, …, 6, 4, 2` (vòng khép lại bằng cạnh `2 — 1`).

Mỗi cặp kề là `(i, i+2)`, cộng thêm `(n−1, n)` ở đỉnh núi và `(1, 2)` ở chân núi.

Với **đường đi**, ta cắt vòng ở cạnh **rẻ nhất** — chính là `(1, 2)` — được dãy `1, 3, …, n, …, 4, 2`.

Tính tay `n = 4` (đường): `1, 3, 4, 2` ⇒ `3 + 12 + 8 = 23` ✓. `n = 6` (vòng): `1,3,5,6,4,2` ⇒ `3 + 15 + 30 + 24 + 8 + 2 = 82` ✓.

In [6]:
def mountain(n):
    odds = list(range(1, n + 1, 2))
    evens = list(range(2, n + 1, 2))
    return odds + evens[::-1]

for n in [4, 5, 6, 7]:
    o = mountain(n)
    print(n, o, '| đường:', score_of(o, False), '| vòng:', score_of(o, True))

4 [1, 3, 4, 2] | đường: 23 | vòng: 25
5 [1, 3, 5, 4, 2] | đường: 46 | vòng: 48
6 [1, 3, 5, 6, 4, 2] | đường: 80 | vòng: 82
7 [1, 3, 5, 7, 6, 4, 2] | đường: 127 | vòng: 129


## 5. Cách 1 — Brute Force: thử mọi hoán vị gán giá trị cho đỉnh

Duyệt mọi hoán vị `1..n` gán cho các đỉnh, tính tổng theo **đúng danh sách cạnh** (không dùng nhận xét đường/vòng). Dùng làm oracle với `n ≤ 8`.

In [7]:
from itertools import permutations

def max_score_brute(n, edges):
    best = 0
    for perm in permutations(range(1, n + 1)):
        best = max(best, sum(perm[a] * perm[b] for a, b in edges))
    return best

for n, e, exp in examples:
    print(max_score_brute(n, e), '| kỳ vọng', exp)

23 | kỳ vọng 23
82 | kỳ vọng 82


**Độ phức tạp:** `O(n! · m)` — không dùng được khi `n` lớn.

**Nút thắt:** đa số hoán vị cho cùng một "cách xếp quanh vòng"; và cách xếp tối ưu có cấu trúc rất đều (dãy núi).

## 6. Ý tưởng thuật toán / bước đột phá

**Lập luận trao đổi (2-opt).** Trên vòng `… a, b, …, c, d …`, đảo ngược đoạn `b … c` đổi hai cạnh `(a,b), (c,d)` thành `(a,c), (b,d)`; các tích bên trong giữ nguyên. Thay đổi điểm:

`Δ = ac + bd − ab − cd = (a − d)(c − b)`.

Cách xếp tối ưu không còn phép đảo nào làm tăng điểm. Trực giác: các số kề nhau nên **chênh lệch ít** (≈ 2), và số lớn kề số lớn — chính là dãy núi. Mục 10 chứng minh chặt chẽ bằng cách đổi sang bài "tổng bình phương hiệu nhỏ nhất" và đếm số lần vòng vượt qua mỗi ngưỡng giá trị; mục 12 kiểm chứng thêm bằng vét cạn với mọi `n ≤ 9`.

**Đường đi:** một cách xếp trên đường là một vòng bị bỏ một cạnh. Điểm đường = điểm vòng − (tích cạnh bị bỏ). Vì vòng núi tối ưu cho vòng và mọi cạnh có tích ≥ `1·2 = 2`, ta có `Điểm(đường) ≤ Vòng_max − 2`, và "vòng núi bỏ cạnh `(1,2)`" đạt đúng chặn đó.

**Thuật toán:** `m == n` ⇒ vòng; ngược lại ⇒ đường; tính điểm dãy núi.

In [8]:
def delta_reverse(a, b, c, d):
    return (a - d) * (c - b)

# ví dụ: vòng [1, 6, 2, 5, 3, 4] xấu; thử đảo đoạn để thấy tăng điểm
o = [1, 6, 2, 5, 3, 4]
print('điểm ban đầu:', score_of(o, True))
i, j = 1, 2                           # đảo đoạn o[1..2]
a, b, c, d = o[i - 1], o[i], o[j], o[j + 1]
o2 = o[:i] + o[i:j + 1][::-1] + o[j + 1:]
print('đảo', o[i:j + 1], '-> Δ dự đoán =', delta_reverse(a, b, c, d), '| điểm mới:', score_of(o2, True))

điểm ban đầu: 59
đảo [6, 2] -> Δ dự đoán = 16 | điểm mới: 75


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Leo đồi bằng phép đảo đoạn (2-opt) — minh họa, không bảo đảm tối ưu

Bắt đầu từ một vòng bất kỳ, liên tục áp dụng phép đảo đoạn có `Δ > 0`. Thường hội tụ tới dãy núi; nhưng tốn `O(n^2)` mỗi vòng lặp và về lý thuyết có thể kẹt ở cực trị địa phương — chỉ để trực quan.

In [9]:
def two_opt_cycle(n, seed=0):
    import random
    rnd = random.Random(seed)
    o = list(range(1, n + 1))
    rnd.shuffle(o)
    improved = True
    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                a, b, c, d = o[i - 1], o[i], o[j], o[(j + 1) % n]
                if (a - d) * (c - b) > 0:
                    o[i:j + 1] = o[i:j + 1][::-1]
                    improved = True
    return score_of(o, True), o

for n in [6, 8, 10]:
    s, o = two_opt_cycle(n, seed=n)
    print(n, 'leo đồi:', s, '| dãy núi:', score_of(mountain(n), True), '| vòng tìm được:', o)

6 leo đồi: 82 | dãy núi: 82 | vòng tìm được: [3, 1, 2, 4, 6, 5]
8 leo đồi: 191 | dãy núi: 191 | vòng tìm được: [6, 8, 7, 5, 3, 1, 2, 4]
10 leo đồi: 368 | dãy núi: 368 | vòng tìm được: [6, 4, 2, 1, 3, 5, 7, 9, 10, 8]


### 7.2. Cách 3 — Tham lam xây dãy núi bằng `deque` (O(n))

Đặt `n` ở giữa; lần lượt `n−1, n−2, …, 1` luân phiên thêm vào bên phải / bên trái. Tính điểm theo đường hoặc vòng.

In [10]:
def max_score_greedy(n, edges):
    dq = deque([n])
    for k, x in enumerate(range(n - 1, 0, -1)):
        if k % 2 == 0:
            dq.append(x)
        else:
            dq.appendleft(x)
    o = list(dq)
    return score_of(o, len(edges) == n)

for n, e, exp in examples:
    print(max_score_greedy(n, e), '| kỳ vọng', exp)

23 | kỳ vọng 23
82 | kỳ vọng 82


Với đường đi, hai đầu của `deque` là `1` và `2` (số nhỏ nhất ở hai mút) — đúng dãy núi bỏ cạnh `(1, 2)`.

### 7.3. Cách 4 — Vòng lặp `Σ i(i+2)` (O(n), lời giải chính dễ đọc)

Các cạnh của vòng núi: `(i, i+2)` với `i = 1..n−2`, cộng `(1, 2)` và `(n−1, n)`. Đường đi: bỏ `(1, 2)`.

In [11]:
def max_score_sum(n, edges):
    if n == 1:
        return 0
    s = sum(i * (i + 2) for i in range(1, n - 1)) + (n - 1) * n
    if len(edges) == n:
        s += 2               # cạnh (1, 2) của vòng
    return s

for n, e, exp in examples:
    print(max_score_sum(n, e), '| kỳ vọng', exp)

23 | kỳ vọng 23
82 | kỳ vọng 82


Lưu ý `n = 2`: tổng rỗng + `1·2 = 2`, đúng (một cạnh). Với `n = 3` vòng: `1·3 + 2·3 + 2 = 11` = `1·2 + 2·3 + 3·1` ✓.

### 7.4. Cách 5 — Công thức đóng O(1)

Với `k = n − 2`: `Σ_{i=1}^{k} i(i+2) = k(k+1)(2k+1)/6 + k(k+1)`.

In [12]:
def max_score_formula(n, edges):
    if n == 1:
        return 0
    k = n - 2
    s = k * (k + 1) * (2 * k + 1) // 6 + k * (k + 1) + (n - 1) * n
    return s + (2 if len(edges) == n else 0)

for n, e, exp in examples:
    print(max_score_formula(n, e), '| kỳ vọng', exp)

23 | kỳ vọng 23
82 | kỳ vọng 82


## 8. TRACE từng bước

Trace cách 3 (xây `deque`) với `n = 7`, cả trường hợp đường và vòng: in trạng thái `deque` sau mỗi bước và từng tích cạnh.

In [13]:
def trace(n, cycle):
    dq = deque([n])
    print(f"bắt đầu: {list(dq)}")
    for k, x in enumerate(range(n - 1, 0, -1)):
        side = 'phải' if k % 2 == 0 else 'trái'
        (dq.append if k % 2 == 0 else dq.appendleft)(x)
        print(f"  thêm {x} vào bên {side}: {list(dq)}")
    o = list(dq)
    prods = [(o[i], o[i + 1], o[i] * o[i + 1]) for i in range(n - 1)]
    if cycle:
        prods.append((o[-1], o[0], o[-1] * o[0]))
    print('  các cạnh:', ', '.join(f"{a}·{b}={p}" for a, b, p in prods))
    print('  tổng =', sum(p for _, _, p in prods), '(vòng)' if cycle else '(đường)')

trace(7, False)
print()
trace(7, True)

bắt đầu: [7]
  thêm 6 vào bên phải: [7, 6]
  thêm 5 vào bên trái: [5, 7, 6]
  thêm 4 vào bên phải: [5, 7, 6, 4]
  thêm 3 vào bên trái: [3, 5, 7, 6, 4]
  thêm 2 vào bên phải: [3, 5, 7, 6, 4, 2]
  thêm 1 vào bên trái: [1, 3, 5, 7, 6, 4, 2]
  các cạnh: 1·3=3, 3·5=15, 5·7=35, 7·6=42, 6·4=24, 4·2=8
  tổng = 127 (đường)

bắt đầu: [7]
  thêm 6 vào bên phải: [7, 6]
  thêm 5 vào bên trái: [5, 7, 6]
  thêm 4 vào bên phải: [5, 7, 6, 4]
  thêm 3 vào bên trái: [3, 5, 7, 6, 4]
  thêm 2 vào bên phải: [3, 5, 7, 6, 4, 2]
  thêm 1 vào bên trái: [1, 3, 5, 7, 6, 4, 2]
  các cạnh: 1·3=3, 3·5=15, 5·7=35, 7·6=42, 6·4=24, 4·2=8, 2·1=2
  tổng = 129 (vòng)


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxScore(self, n: int, edges: List[List[int]]) -> int:
        if n == 1:
            return 0
        k = n - 2
        s = k * (k + 1) * (2 * k + 1) // 6 + k * (k + 1) + (n - 1) * n
        if len(edges) == n:          # chu trình: thêm cạnh 1·2
            s += 2
        return s

for n, e, exp in examples:
    got = Solution().maxScore(n, e)
    print(got, 'OK' if got == exp else 'SAI')

23 OK
82 OK


## 10. Vì sao thuật toán đúng

**Bước 1 — đổi bài "tích lớn nhất" thành "hiệu bình phương nhỏ nhất".** Trên vòng, mỗi số xuất hiện ở đúng hai cạnh, nên

`Σ_cạnh (a − b)² = Σ_cạnh (a² + b²) − 2 Σ_cạnh ab = 2·Σ_{i=1}^{n} i² − 2·Điểm`.

⇒ **Điểm lớn nhất ⇔ tổng bình phương hiệu các cạnh `Σ d²` nhỏ nhất** (với `d = |a − b|`).

**Bước 2 — chặn dưới `Σ d`.** Với mỗi ngưỡng `k ∈ [1, n−1]`, chia số thành `≤ k` và `> k`. Vòng phải đi qua ranh giới này **ít nhất hai lần** (đi sang rồi quay về). Một cạnh có hiệu `d` vượt đúng `d` ngưỡng ⇒ `Σ d ≥ 2(n − 1)`.

**Bước 3 — chặn dưới `Σ d²`.** Vì `(d − 2)² ≥ 0` nên `d² ≥ 4d − 4`, dấu bằng ⇔ `d = 2`. Cộng trên `n` cạnh: `Σ d² ≥ 4Σd − 4n + Σ(d − 2)² ≥ 8(n−1) − 4n + Σ(d−2)² = 4n − 8 + Σ(d−2)²`.

Với `n ≥ 5`: hai cạnh tại đỉnh mang số 1 có hai hiệu **khác nhau** (hai hàng xóm khác nhau) nên ít nhất một cạnh có `d ≠ 2`, góp `(d−2)² ≥ 1`; tương tự tại số `n`. Nếu hai cạnh "xấu" đó trùng nhau thì đó là cạnh `(1, n)` với `d = n − 1 ≥ 4`, góp `≥ 4`. Vậy `Σ(d−2)² ≥ 2` ⇒ `Σ d² ≥ 4n − 6`.

**Bước 4 — dãy núi đạt chặn.** Dãy núi có `n − 2` cạnh `d = 2` và hai cạnh `d = 1` (`1–2`, `(n−1)–n`): `Σ d² = 4(n − 2) + 2 = 4n − 6`. ⇒ tối ưu cho vòng (`n ≤ 4` kiểm bằng vét cạn).

**Bước 5 — đường đi.** Nối hai mút `x, y` của một đường được một vòng: `Điểm(đường) = Điểm(vòng) − x·y ≤ Vòng_max − 2` (vì `x ≠ y` ⇒ `x·y ≥ 2`). Dãy núi bỏ cạnh `1–2` đạt đúng `Vòng_max − 2`.

**Bước 6 — công thức:** cạnh của vòng núi là `(i, i+2)` với `i = 1..n−2`, cộng `(1,2)` và `(n−1, n)`.

In [15]:
# Kiểm tra đẳng thức của bước 1 và chặn dưới của bước 3 trên nhiều vòng ngẫu nhiên
random_mod = __import__('random')
rnd = random_mod.Random(10)
for n in range(5, 12):
    sq = sum(i * i for i in range(1, n + 1))
    for _ in range(200):
        o = list(range(1, n + 1)); rnd.shuffle(o)
        d2 = sum((o[i] - o[(i + 1) % n]) ** 2 for i in range(n))
        assert d2 == 2 * sq - 2 * score_of(o, True)
        assert d2 >= 4 * n - 6
    m = mountain(n)
    assert sum((m[i] - m[(i + 1) % n]) ** 2 for i in range(n)) == 4 * n - 6
print('Đẳng thức Σd² = 2Σi² − 2·Điểm và chặn Σd² ≥ 4n − 6 đều đúng; dãy núi đạt dấu bằng.')

Đẳng thức Σd² = 2Σi² − 2·Điểm và chặn Σd² ≥ 4n − 6 đều đúng; dãy núi đạt dấu bằng.


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn hoán vị | `O(n! · m)` | `O(n)` |
| 2. Leo đồi 2-opt | `O(n^2)` mỗi lượt cải thiện (không bảo đảm) | `O(n)` |
| 3. Deque tham lam | `O(n)` | `O(n)` |
| 4. Vòng lặp `Σ i(i+2)` | `O(n)` | `O(1)` |
| 5. Công thức đóng | `O(1)` (bỏ qua việc đọc `edges`) | `O(1)` |

Kết quả có thể tới `~n³/3 ≈ 4·10^{13}` với `n = 5·10^4` — Python không tràn số; ngôn ngữ khác cần `long long`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [16]:
impls = {
    'greedy': max_score_greedy,
    'sum': max_score_sum,
    'formula': max_score_formula,
    'submit': lambda n, e: Solution().maxScore(n, e),
}
edge = [
    (2, [[0, 1]], 2),
    (3, [[0, 1], [1, 2]], 9),            # 1,3,2 -> 3 + 6
    (3, [[0, 1], [1, 2], [2, 0]], 11),   # tam giác: 2 + 6 + 3
    (5, [[0, 1], [1, 2], [2, 3], [3, 4], [4, 0]], 1 * 3 + 3 * 5 + 5 * 4 + 4 * 2 + 2 * 1),
]
for n, e, exp in examples + edge:
    for name, f in impls.items():
        got = f(n, e)
        assert got == exp, (name, n, e, got, exp)
    assert max_score_brute(n, e) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu với vét cạn cho mọi `n ≤ 9`, cả đường và vòng, đánh số đỉnh ngẫu nhiên

In [17]:
import random
random.seed(3547)

def rand_graph(n, cycle):
    perm = list(range(n))
    random.shuffle(perm)
    es = [[perm[i], perm[i + 1]] for i in range(n - 1)]
    if cycle and n >= 3:
        es.append([perm[-1], perm[0]])
    random.shuffle(es)
    return [e[::-1] if random.random() < 0.5 else e for e in es]

cnt = 0
for n in range(2, 10):
    for cycle in (False, True):
        if cycle and n < 3:
            continue
        e = rand_graph(n, cycle)
        exp = max_score_brute(n, e)
        for name, f in impls.items():
            assert f(n, e) == exp, (name, n, cycle)
        cnt += 1
print('Khớp vét cạn cho', cnt, 'đồ thị (n = 2..9, đường và vòng).')

Khớp vét cạn cho 15 đồ thị (n = 2..9, đường và vòng).


### 12.3. Ca lớn (`n = 5·10^4`)

In [18]:
import time
n = 5 * 10**4
for cycle in (False, True):
    e = rand_graph(n, cycle)
    t = time.time(); a = Solution().maxScore(n, e); t1 = time.time() - t
    b = max_score_greedy(n, e)
    assert a == b == max_score_sum(n, e)
    print(f"{'vòng' if cycle else 'đường'}: {a} | {t1 * 1e6:.1f} µs | vượt 2^31? {a > 2**31}")

đường: 41667916575001 | 12.2 µs | vượt 2^31? True
vòng: 41667916575003 | 10.7 µs | vượt 2^31? True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Xếp tăng dần `1, 2, 3, …, n` dọc đường/vòng

In [19]:
def wrong_sorted(n, edges):
    o = list(range(1, n + 1))
    return score_of(o, len(edges) == n)

for n, e, exp in examples:
    print('sai :', wrong_sorted(n, e), '| đúng:', Solution().maxScore(n, e))

sai : 20 | đúng: 23
sai : 76 | đúng: 82


**Vì sao sai:** trên vòng, xếp tăng dần tạo cạnh `n·1` lãng phí số lớn nhất; trên đường, số `n` ở mút chỉ có một hàng xóm. Cách tối ưu đặt `n` ở giữa với hai hàng xóm lớn.

### Lỗi 2 — Với đường đi, dùng luôn điểm của vòng (quên trừ cạnh `1·2`)

In [20]:
def wrong_path_as_cycle(n, edges):
    return score_of(mountain(n), True)

n, e, exp = examples[0]
print('sai :', wrong_path_as_cycle(n, e), '| đúng:', Solution().maxScore(n, e))

sai : 25 | đúng: 23


**Vì sao sai:** đường có `n − 1` cạnh; không có cạnh nối hai mút.

### Lỗi 3 — Gán theo bậc: số lớn cho đỉnh bậc 2, nhưng đặt số lớn xen kẽ số nhỏ

In [21]:
def wrong_alternate(n, edges):
    # SAI: xen kẽ lớn - nhỏ (n, 1, n-1, 2, ...) với hi vọng "cân bằng"
    lo, hi, o = 1, n, []
    while lo <= hi:
        o.append(hi); hi -= 1
        if lo <= hi:
            o.append(lo); lo += 1
    return score_of(o, len(edges) == n)

for n, e, exp in examples:
    print('sai :', wrong_alternate(n, e), '| đúng:', Solution().maxScore(n, e))

sai : 13 | đúng: 23
sai : 59 | đúng: 82


**Vì sao sai:** xen kẽ làm số lớn luôn nhân với số nhỏ — gần như ngược với mục tiêu. Muốn lớn nhất thì số lớn phải kề số lớn (mục 10: các hiệu kề phải nhỏ, ≈ 2).

### Lỗi 4 — Dùng kiểu số 32-bit (trong ngôn ngữ khác) hoặc lấy modulo không cần thiết

In [22]:
n = 5 * 10**4
v = Solution().maxScore(n, rand_graph(n, True))
print('giá trị thật:', v, '| ép về int32:', (v + 2**31) % 2**32 - 2**31)

giá trị thật: 41667916575003 | ép về int32: -1856130789


**Vì sao sai:** đáp án cỡ `4·10^{13}` vượt `2^{31}`; đề không yêu cầu modulo. Trong Python không lo, nhưng khi dịch sang C++/Java phải dùng `long long`/`long`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đồ thị liên thông, bậc ≤ 2" ⇒ chỉ là **đường** hoặc **vòng**; thường chỉ cần `m == n`.
- "Gán hoán vị `1..n` để tối đa tổng tích kề" ⇒ **dãy núi** (lẻ tăng, chẵn giảm) — số lớn kề số lớn.
- "Tối thiểu tổng tích kề": xen kẽ lớn–nhỏ là trực giác tốt nhưng **không** luôn tối ưu (bài tập 4 cho thấy sai từ `n = 5`) — đừng tin trực giác khi chưa kiểm chứng.
- Lập luận đảo đoạn: `Δ = (a − d)(c − b)` — công cụ chung cho bài sắp xếp trên vòng.
- Tổng có cấu trúc `Σ i(i+k)` ⇒ dùng công thức tổng bình phương để ra `O(1)`.

## 15. Biến thể của bài

1. **Tối thiểu** tổng tích kề trên vòng ⇒ khó hơn hẳn: xen kẽ lớn–nhỏ chỉ là xấp xỉ (bài tập 4).
2. Giá trị cho trước là một **multiset bất kỳ** (không phải `1..n`) ⇒ cùng ý tưởng: sắp xếp, xếp dãy núi.
3. Đồ thị là **nhiều thành phần** đường/vòng ⇒ chia các số lớn cho thành phần nào? (bài khó hơn).
4. Tối đa tổng **|hiệu|** kề nhau ⇒ xếp xen kẽ lớn–nhỏ.

In [23]:
# Biến thể 2: multiset bất kỳ — xếp dãy núi trên giá trị đã sắp
def max_cycle_multiset(vals):
    v = sorted(vals)
    o = v[0::2] + v[1::2][::-1]
    return score_of(o, True)

def brute_cycle_multiset(vals):
    return max(score_of(list(p), True) for p in permutations(vals))

random.seed(2)
for _ in range(100):
    vals = [random.randint(1, 20) for _ in range(random.randint(3, 7))]
    assert max_cycle_multiset(vals) == brute_cycle_multiset(vals)
print('Biến thể 2 khớp vét cạn; ví dụ:', max_cycle_multiset([10, 1, 7, 3, 3]))

Biến thể 2 khớp vét cạn; ví dụ: 127


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn hoán vị | `O(n! m)` | `O(n)` | Oracle `n ≤ 9` |
| 2 | Leo đồi 2-opt | `O(n^2)`/lượt | `O(n)` | Trực quan hóa chứng minh |
| 3 | Deque dãy núi | `O(n)` | `O(n)` | Khi cần chính cách gán |
| 4 | Vòng lặp tổng | `O(n)` | `O(1)` | Dễ đọc |
| 5 | Công thức đóng | `O(1)` | `O(1)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxScore(5, đường)` = ?

**Bài 2 (tính tay):** `maxScore(4, vòng)` = ?

**Bài 3 (điền TODO):** viết `assignment(n, edges)` trả về giá trị gán cho **từng đỉnh** (theo chỉ số đỉnh) đạt tối ưu.

**Bài 4 (kiểm tra giả thuyết):** biến thể 1 — giả thuyết "xen kẽ lớn–nhỏ cho tổng tích kề **nhỏ nhất** trên vòng `1..n`". Viết code kiểm tra giả thuyết bằng vét cạn.

**Bài 5 (mini challenge):** chứng minh bằng máy: với mọi `n ≤ 8`, mọi vòng tối ưu (vét cạn) đều là dãy núi (sai khác bởi xoay/lật).

In [24]:
def assignment_todo(n, edges):
    # TODO: tìm thứ tự đỉnh dọc đường/vòng, gán theo dãy núi
    raise NotImplementedError

def min_cycle_todo(n):
    # TODO
    raise NotImplementedError

def all_optimal_are_mountains_todo(n):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy `1,3,5,4,2`. (2) `3 + 15 + 20 + 8`. (3) `46`.

**Bài 2.** (1) Vòng `1,3,4,2`. (2) `3 + 12 + 8 + 2`. (3) `25`.

**Bài 3.** (1) Tìm một mút (đỉnh bậc 1) với đường, hoặc đỉnh bất kỳ với vòng. (2) Đi dọc đồ thị để có thứ tự đỉnh. (3) Gán `mountain(n)[i]` cho đỉnh thứ `i`, kiểm tra tổng.

**Bài 4.** (1) Xen kẽ lớn–nhỏ. (2) So vét cạn cho `n ≤ 8`. (3) Nếu không khớp, thử nhiều cách xen kẽ và lấy min (đây là bài kiểm chứng giả thuyết).

**Bài 5.** (1) Liệt kê mọi hoán vị đạt max. (2) Chuẩn hóa: xoay để `1` đứng đầu và chọn chiều có phần tử thứ hai nhỏ hơn. (3) So với `mountain(n)` đã chuẩn hóa.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
path5 = [[i, i + 1] for i in range(4)]
cyc4 = [[0, 1], [1, 2], [2, 3], [3, 0]]
a1, a2 = Solution().maxScore(5, path5), Solution().maxScore(4, cyc4)
assert a1 == 46 == max_score_brute(5, path5)
assert a2 == 25 == max_score_brute(4, cyc4)
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 46 | Bài 2: 25


In [26]:
# Bài 3: gán giá trị cho từng đỉnh
def assignment(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    start = next((v for v in range(n) if len(adj[v]) <= 1), 0)   # mút của đường (hoặc 0 nếu là vòng)
    order, seen, cur = [start], {start}, start
    while len(order) < n:
        cur = next(x for x in adj[cur] if x not in seen)
        seen.add(cur)
        order.append(cur)
    val = [0] * n
    for v, x in zip(order, mountain(n)):
        val[v] = x
    return val

random.seed(3)
for n in range(2, 10):
    for cycle in (False, True):
        if cycle and n < 3:
            continue
        e = rand_graph(n, cycle)
        val = assignment(n, e)
        assert sorted(val) == list(range(1, n + 1))
        assert sum(val[a] * val[b] for a, b in e) == Solution().maxScore(n, e)
print('Bài 3: ví dụ 2 ->', assignment(*examples[1][:2]))

Bài 3: ví dụ 2 -> [1, 5, 2, 3, 4, 6]


In [27]:
# Bài 4: tối thiểu trên vòng
def min_cycle(n):
    lo, hi, o = 1, n, []
    while lo <= hi:
        o.append(hi); hi -= 1
        if lo <= hi:
            o.append(lo); lo += 1
    return score_of(o, True)

def min_cycle_brute(n):
    return min(score_of([1] + list(p), True) for p in permutations(range(2, n + 1)))

res = [(n, min_cycle(n), min_cycle_brute(n)) for n in range(3, 9)]
print('n, xen kẽ, vét cạn:', res)
print('xen kẽ lớn-nhỏ có luôn tối ưu?', all(a == b for _, a, b in res))

n, xen kẽ, vét cạn: [(3, 11, 11), (4, 21, 21), (5, 38, 37), (6, 59, 58), (7, 90, 87), (8, 126, 123)]
xen kẽ lớn-nhỏ có luôn tối ưu? False


In [28]:
# Bài 5: mọi vòng tối ưu đều là dãy núi (sai khác xoay/lật)
def canon(o):
    i = o.index(1)
    r = o[i:] + o[:i]
    return min(r, [r[0]] + r[1:][::-1])

def all_optimal_are_mountains(n):
    best = max(score_of([1] + list(p), True) for p in permutations(range(2, n + 1)))
    opts = {tuple(canon([1] + list(p))) for p in permutations(range(2, n + 1))
            if score_of([1] + list(p), True) == best}
    return opts == {tuple(canon(mountain(n)))}

print({n: all_optimal_are_mountains(n) for n in range(3, 9)})

{3: True, 4: True, 5: True, 6: True, 7: True, 8: True}


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
vòng núi: 1, 3, 5, ..., n, ..., 6, 4, 2   (cạnh (i, i+2), (n-1, n), (1, 2))
k = n - 2
S = k(k+1)(2k+1)/6 + k(k+1) + n(n-1)      # đường đi
nếu m == n (vòng): S += 2
```

**Checklist:**

- [ ] Phân biệt đường (`m = n − 1`) và vòng (`m = n`)?
- [ ] Số lớn kề số lớn (dãy núi), không xếp tăng dần?
- [ ] Đường: bỏ cạnh `1·2`?
- [ ] `n = 1`, `n = 2` đúng?
- [ ] Kết quả có thể vượt 32-bit?